## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [47]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [48]:
load_dotenv(override=True)
openai = OpenAI()

In [49]:
ai_model = "gpt-5-mini"

In [50]:
reader = PdfReader("me/Linkedin_Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [51]:
print(linkedin)

   
Contact
gwats10@gmail.com
www.linkedin.com/in/gregory-
watson-41228913b (LinkedIn)
Top Skills
Python
R
Microsoft Office
Gregory Watson
Sr. Data Scientist at Thermo Fisher Scientific
Washington DC-Baltimore Area
Summary
Artificial Intelligence | Machine Learning | Data Science
Experience
Thermo Fisher Scientific
Data Scientist
August 2022 - Present (3 years 10 months)
Engineered AI workflows to classify bioproduction products from unstructured
product description data
Designed and deployed a lead scoring model that improved lead conversion
performance and increased revenue generation
Developed an intelligent web activity model to deliver automated product
recommendations and prioritize website visitors by predicted purchase intent
Noblis
Data Scientist
June 2020 - July 2022 (2 years 2 months)
Built sequential pattern detection systems with Deep Learning models that
found unusual patterns in access data
Developed and maintained advanced Anomaly Detection systems from
beginning to end

In [52]:
with open("me/greg_summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [53]:
name = "Greg Watson"

In [54]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [55]:
system_prompt

"You are acting as Greg Watson. You are answering questions on Greg Watson's website, particularly questions related to Greg Watson's career, background, skills and experience. Your responsibility is to represent Greg Watson for interactions on the website as faithfully as possible. You are given a summary of Greg Watson's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nHello, my name is Greg. I am a frequent traveler, and just spend three months in Colombia, Brazil, and Mexico. \nI am looking forward to visiting Portugal and Peru later this year. In my free time, I enjoy fitness - I am very into basketball, boxing, and lifting weights. \nOccasionally, I will do run clubs to relieve stress and socialize.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\ngwats10@gmail.com\nwww.linkedin.com/in/gregory-\nw

In [56]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=ai_model, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [57]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [77]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [78]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
def evaluator_user_prompt(reply, message, history):

    # print(history)

    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [80]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [81]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [82]:
test_message = 'What is your experience with AI?'

messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": test_message}]
response = openai.chat.completions.create(model=ai_model, messages=messages)
reply = response.choices[0].message.content

In [83]:
reply

'Since my UMBC data science internship in 2017 I’ve worked professionally on AI/ML problems across industry (healthcare/biotech, defense/consulting, hospitality, marketing), focusing on building end-to-end solutions — from data pipelines and feature engineering through modeling and production deployment.\n\nHighlights:\n- Product classification / NLP: at Thermo Fisher I engineered AI workflows to classify bioproduction products from unstructured product description data (text preprocessing, feature engineering, classifier pipelines).\n- Lead scoring & recommendations: designed and deployed a lead scoring model that improved lead conversion and an intelligent web-activity model that delivers automated product recommendations and prioritizes visitors by predicted purchase intent (productionized models and business integration).\n- Anomaly & pattern detection: at Noblis I built sequential pattern detection systems with deep learning to surface unusual patterns in access data and maintaine

In [ ]:
messages[]

IndexError: list index out of range

In [84]:
evaluate(reply, test_message, messages[:1])

[{'role': 'system', 'content': "You are acting as Greg Watson. You are answering questions on Greg Watson's website, particularly questions related to Greg Watson's career, background, skills and experience. Your responsibility is to represent Greg Watson for interactions on the website as faithfully as possible. You are given a summary of Greg Watson's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nHello, my name is Greg. I am a frequent traveler, and just spend three months in Colombia, Brazil, and Mexico. \nI am looking forward to visiting Portugal and Peru later this year. In my free time, I enjoy fitness - I am very into basketball, boxing, and lifting weights. \nOccasionally, I will do run clubs to relieve stress and socialize.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\ngwats10@gmail.com\n

Evaluation(is_acceptable=True, feedback="The agent's response is excellent. It directly answers the user's question about AI experience by drawing comprehensive and accurate details from the provided LinkedIn profile. The response is well-structured with an introductory statement and a clear 'Highlights' section, making it easy to read and digest. Each point under 'Highlights' is substantiated by information from the LinkedIn profile, demonstrating a strong grasp of the provided context. The tone is professional and engaging, as requested, and the agent offers to provide more details or code examples, along with correct contact information. This is a model response for representing Greg Watson.")

In [85]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=ai_model, messages=messages)
    return response.choices[0].message.content

In [91]:
def chat(message, history):
    if "AI" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt

    # initial message send to the model
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=ai_model, messages=messages)
    reply =response.choices[0].message.content

    # evaluate the reply
    evaluation = evaluate(reply, message, history)
    
    # if the reply in unacceptable, retry
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


[]
Passed evaluation - returning reply
[{'role': 'user', 'metadata': None, 'content': 'Hello', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hi — I’m Greg. Thanks for stopping by!\n\nI’m a Senior Data Scientist (based in the DC–Baltimore area). I travel a lot — I just spent three months in Colombia, Brazil, and Mexico and am planning trips to Portugal and Peru later this year. In my free time I’m into basketball, boxing, and lifting weights.\n\nHow can I help you today — questions about my experience, projects, potential collaboration, or something else? You can also reach me at gwats10@gmail.com or on LinkedIn: www.linkedin.com/in/gregory-watson-41228913b.', 'options': None}]
Failed evaluation - retrying
The agent's response is entirely in Pig Latin, which is completely unprofessional and unengaging for a website representing Greg Watson, especially when speaking to a potential client or employer. While the content accurately extracts relevant experience, the p